In [169]:
import pandas as pd
import numpy as np
from pathlib import Path

random_seed = 42
rng = np.random.default_rng(random_seed)

In [170]:
N_sites= 150
site_ids = [f"Site_{i:03d}" for i in range (1, N_sites +1)]
print(f"N° of synthetic sites: {len(site_ids)}")

N° of synthetic sites: 150


In [171]:
#site characteristics
data = pd.DataFrame({
    "site_id": site_ids,
    "act_delay_days": rng.integers(
        10, 100, size=N_sites
    ),
    "expected_recruit_rate": np.round(
        rng.uniform(0.5, 5.0, size=N_sites), 2
    ),
    "rec_perf_ratio": np.round(
        rng.uniform(0.5, 1.3, size=N_sites), 2
    ),
    "op_reliab": np.round(
        rng.uniform(0.60,0.99, size=N_sites), 2
    ),
    "startup_cost": rng.integers(
        5000, 20001, size=N_sites
    ),
    "month_op_cost": rng.integers(
        2000, 8000, size=N_sites
    )
})

In [172]:
# Normalise operational characteristics to a 0–1 scale
urgency = (90 - data["act_delay_days"]) / 90

reliability_score = (
    (data["op_reliab"] - 0.60) / (0.99 - 0.60)
)

# Generate costs with operationally meaningful relationships
data["startup_cost"] = (
    5000
    + urgency * 8000
    + reliability_score * 4000
    + np.random.normal(0, 1200, len(data))
).round().clip(5000, 20000).astype(int)

data["month_op_cost"] = (
    2000
    + urgency * 1800
    + reliability_score * 1000
    + np.random.normal(0, 500, len(data))
).round().clip(2000, 8000).astype(int)

In [173]:
#test
print("Ds Shape:", data.shape)
print("Missing values:")
print(data.isna().sum())
print("summary stats")
display(data.describe().round(2))

Ds Shape: (150, 7)
Missing values:
site_id                  0
act_delay_days           0
expected_recruit_rate    0
rec_perf_ratio           0
op_reliab                0
startup_cost             0
month_op_cost            0
dtype: int64
summary stats


,act_delay_days,expected_recruit_rate,rec_perf_ratio,op_reliab,startup_cost,month_op_cost
count,150.00,150.00,150.00,150.00,150.00,150.00
mean,55.86,2.69,0.89,0.80,9825.33,3178.08
std,24.79,1.30,0.23,0.11,2669.68,738.26
min,10.00,0.56,0.51,0.60,5000.00,2000.00
25%,34.25,1.55,0.69,0.69,7811.25,2627.25
50%,58.00,2.70,0.90,0.80,9415.00,3145.00
75%,78.00,3.76,1.12,0.89,11856.75,3716.00
max,99.00,4.97,1.28,0.99,15315.00,5906.00


In [174]:
#controlpoint
assert data["expected_recruit_rate"].between(0.5, 5.0).all()
assert data["rec_perf_ratio"].between(0.5, 1.3).all()
assert data["op_reliab"].between(0.60, 0.99).all()
assert data["startup_cost"].between(5000, 20000).all()
assert data["month_op_cost"].between(2000, 8000).all()

print("All variable ranges are valid.")


print("Missing values:", data.isna().sum().sum())
print("Duplicate site IDs:", data["site_id"].duplicated().sum())
print("Dataset dimensions:", data.shape)

All variable ranges are valid.
Missing values: 0
Duplicate site IDs: 0
Dataset dimensions: (150, 7)


In [175]:
data.to_csv("../data/sites_data.csv", index=False)


print("DS exported succesfully")

DS exported succesfully
